# 05 · Ingestão Bronze — Squad 3 | Henrique Ficht
Lê os CSVs históricos da camada **raw** (ADLS Gen2), adiciona colunas de
auditoria (`bronze_ingested_at`, `bronze_source_file`), e salva como
**Delta** na camada **bronze**, particionado por `ano` e `mes`.

- **Tabelas:** `food_fornecedores` · `food_lotes_producao`
- **Modo de escrita:** `append` (acumulativo — silver fará o merge)
- **Frequência:** semanal (toda segunda-feira)

In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
# ── Configuração de credenciais e paths ────────────────────────────────────
import os
from dotenv import load_dotenv

ENV_PATH = "/Workspace/Users/ik.kukoo@gmail.com/.env"
load_dotenv(ENV_PATH, override=True)

def _require(key):
    value = os.getenv(key)
    if not value:
        raise EnvironmentError(f"Variável '{key}' não encontrada em {ENV_PATH}")
    return value

STORAGE_ACCOUNT = "internshipdatalake"

# ── Autenticação OAuth (Service Principal) ──────────────────────────────────────────
adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{_require('ADLS_TENANT_ID')}/oauth2/token",
}

# ── Paths ──────────────────────────────────────────────────────────────────
# Origem: container raw
RAW_BASE = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data"

# Destino: container bronze (mesmo nível que raw no storage account)
BRONZE_BASE = f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"

print("✅ Config carregado.")
print(f"   RAW    → {RAW_BASE}")
print(f"   BRONZE → {BRONZE_BASE}")

In [0]:
# ── Verificar existência e acesso ao container 'bronze' ─────────────────────

try:
    test_path = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    spark.read.format("binaryFile").options(**adls_options).load(test_path)
    print("✅ Container 'bronze' acessível via Service Principal.")
except Exception as e:
    error_msg = str(e)
    if "ContainerNotFound" in error_msg:
        print("⚠️  Container 'bronze' NÃO EXISTE no storage account.")
    elif "403" in error_msg or "AuthorizationPermissionMismatch" in error_msg:
        print("⚠️  Container 'bronze' existe, mas Service Principal SEM permissão.")
    else:
        print(f"❌ Erro inesperado: {e}")

In [0]:
# ── Função genérica de ingestão Bronze ────────────────────────────────────
from pyspark.sql.functions import current_timestamp, year, month, col

def ingest_bronze(table_name: str, partition_date_col: str):
    """
    Lê um CSV da camada raw, adiciona colunas de auditoria e
    salva como Delta na camada bronze, particionado por ano/mês.
    """
    raw_path    = f"{RAW_BASE}/{table_name}.csv"
    bronze_path = f"{BRONZE_BASE}/{table_name}"

    print(f"\n{'='*60}")
    print(f"📥 Ingestão Bronze: {table_name}")
    print(f"   Origem:  {raw_path}")
    print(f"   Destino: {bronze_path}")
    print(f"   Partição por: {partition_date_col} → ano/mes")
    print(f"{'='*60}")

    # 1. Ler CSV da camada raw
    df = (
        spark.read
        .options(**adls_options)
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(raw_path)
    )
    raw_count = df.count()
    print(f"\n   ✅ Leitura raw: {raw_count} linhas | {len(df.columns)} colunas")

    # 2. Adicionar colunas de auditoria
    df = (
        df
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("bronze_source_file", col("_metadata.file_path"))
    )

    # 3. Criar colunas de partição (ano e mês extraídos da coluna de data)
    df = (
        df
        .withColumn("ano", year(partition_date_col))
        .withColumn("mes", month(partition_date_col))
    )

    # 4. Salvar como Delta na camada bronze (append)
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("ano", "mes")
        .options(**adls_options)
        .save(bronze_path)
    )

    print(f"   ✅ Delta gravado em modo append")

    # 5. Validação pós-gravação
    df_check = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(bronze_path)
    )
    bronze_count = df_check.count()
    print(f"   ✅ Validação: {bronze_count} registros no Delta bronze")
    print(f"   📊 Colunas: {df_check.columns}")

    return df_check

In [0]:
# ── Ingestão: food_fornecedores ───────────────────────────────────────────
# Partição por dt_cadastro (data de cadastro do fornecedor)

df_bronze_fornecedores = ingest_bronze(
    table_name="food_fornecedores",
    partition_date_col="dt_cadastro"
)

df_bronze_fornecedores.printSchema()
df_bronze_fornecedores.show(5, truncate=False)

In [0]:
# ── Ingestão: food_lotes_producao ─────────────────────────────────────────
# Partição por dt_fabricacao (data de fabricação do lote)

df_bronze_lotes = ingest_bronze(
    table_name="food_lotes_producao",
    partition_date_col="dt_fabricacao"
)

df_bronze_lotes.printSchema()
df_bronze_lotes.show(5, truncate=False)

In [0]:
# ── Resumo final ──────────────────────────────────────────────────────────
print("\n🎉 Ingestão Bronze concluída com sucesso!")
print("   RAW (CSV) → BRONZE (Delta + auditoria + partição ano/mes)")
print(f"\n   📁 food_fornecedores:   {df_bronze_fornecedores.count()} registros")
print(f"   📁 food_lotes_producao: {df_bronze_lotes.count()} registros")
print(f"\n   📍 Destino: {BRONZE_BASE}/")
print("   🔄 Modo: append (semanal — toda segunda-feira)")
print("   📊 Auditoria: bronze_ingested_at, bronze_source_file")
print("   📂 Partição: ano, mes")